# Search engine
<hr>

# Install packages

In [0]:
!pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
!pip install googletrans

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
!pip install nltk

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


# Import packages

In [0]:
# basic packages

import pandas as pd
import numpy as np
import re

# transalte

from googletrans import Translator

# NLP 

import nltk
from nltk.corpus import stopwords
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [0]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/spark-71ef0264-7cfc-4c71-a4bd-32/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Retrieval data

In [0]:
df_zeus = pd.read_excel('./data/Repositorio Zeus II.xlsx')

In [0]:
# show data

df_zeus.head()

,N,Autor 1,Autor 2,Tutor,Titulo,Semestre,email,Modalidad,Codigo,Calificacion,Categoria,Palabras clave,Resumen,Enlace tesis,Enlace Resumen,Diapositivas,Simulacion
0,1,DANNY MARIO OROPEZA ROJAS,NO CORRESPONDE,NO HAY REGISTROS,PROYECTO DE FACTIBILIDAD PARA LA IMPLEMENTACIÓ...,I-2004,dannymario_oropeza@hotmail.com,PROYECTO DE GRADO,IND-000-2004,100,FACTIBILIDAD,Biodiesel,El Proyecto de grado tiene como principal obje...,https://drive.google.com/file/d/1yNmUbZT_nqRNB...,https://drive.google.com/file/d/1TlxFBwnFfYsZW...,NaN,No tiene
1,2,ESPINOZA HINOJOSA JOSE ISRAEL,FLORES AGUIRRE WILLY MAGNO,NO HAY REGISTROS,PROYECTO DE FACTIBILIDAD ECONÓMICA Y FINANCIER...,I-2004,No hay registros,PROYECTO DE GRADO,IND-001-2004,100,FACTIBILIDAD,Camu camu,El camu camu es una variedad de fruto que ti...,https://drive.google.com/file/d/1CQw2OpRN_seOj...,No tiene,No tiene,No tiene
2,3,CHIRINOS LOZADA JAVIER SERAPIO,BECERRA DOMINGUEZ YSRAEL,NO HAY REGISTROS,ESTUDIO DE FACTIBILIDAD PARA UNA PLANTA RECICL...,I-2004,ysra12@hotmail.com,PROYECTO DE GRADO,IND-002-2004,100,FACTIBILIDAD,Recicladora,"Se cumplieron los objetivos deseados, en los o...",https://drive.google.com/file/d/1JBMpCG3OQSvTS...,No tiene,No tiene,No tiene
3,4,CARLA VIVIAN ROJAS NAVIA,NO CORRESPONDE,NO HAY REGISTROS,DIAGNOSTICO DEL SISTEMA DE COBRANZA INTERNO DE...,I-2004,carlavivianarn@yahoo.es,TRABAJO DIRIGIDO,IND-003-2004,100,SISTEMAS DE INFORMACIÓN,VIVA,La empresa de telecomunicaciones VIVA-GSM cuen...,https://drive.google.com/file/d/1Q1a9v3pwCCkX8...,No tiene,No tiene,No tiene
4,5,VASQUEZ FLORES YOSHIDA IVETT,NO CORRESPONDE,NO HAY REGISTROS,ESTUDIO DE FACTIBILIDAD PARA LA IMPLEMENTACIÓN...,I-2004,yoshi_112000@yahoo.es,PROYECTO DE GRADO,IND-004-2004,100,FACTIBILIDAD,Yuca,El objetivo principal de este proyecto fue el ...,https://drive.google.com/file/d/1T5xhIdT3JMlGN...,No tiene,No tiene,No tiene


In [0]:
df_zeus.shape

(1172, 17)

# Search engine - titles

In [0]:
df_all_trans = pd.read_excel('./data/summarize_tesis.xlsx')

In [0]:
df_all_trans.head()

,N,Autor 1,Autor 2,Tutor,Titulo,Semestre,email,Modalidad,Codigo,Calificacion,Categoria,Palabras clave,Resumen,Enlace tesis,Enlace Resumen,Diapositivas,Simulacion,thesis_title,summarize_thesis
0,1,DANNY MARIO OROPEZA ROJAS,NO CORRESPONDE,NO HAY REGISTROS,PROYECTO DE FACTIBILIDAD PARA LA IMPLEMENTACIÓ...,I-2004,dannymario_oropeza@hotmail.com,PROYECTO DE GRADO,IND-000-2004,100,FACTIBILIDAD,Biodiesel,El Proyecto de grado tiene como principal obje...,https://drive.google.com/file/d/1yNmUbZT_nqRNB...,https://drive.google.com/file/d/1TlxFBwnFfYsZW...,NaN,No tiene,FEASIBILITY PROJECT FOR THE IMPLEMENTATION OF ...,The main objective of the degree project is to...
1,2,ESPINOZA HINOJOSA JOSE ISRAEL,FLORES AGUIRRE WILLY MAGNO,NO HAY REGISTROS,PROYECTO DE FACTIBILIDAD ECONÓMICA Y FINANCIER...,I-2004,No hay registros,PROYECTO DE GRADO,IND-001-2004,100,FACTIBILIDAD,Camu camu,El camu camu es una variedad de fruto que ti...,https://drive.google.com/file/d/1CQw2OpRN_seOj...,No tiene,No tiene,No tiene,ECONOMIC AND FINANCIAL FEASIBILITY PROJECT FOR...,Camu camu is a variety of fruit that has its o...
2,3,CHIRINOS LOZADA JAVIER SERAPIO,BECERRA DOMINGUEZ YSRAEL,NO HAY REGISTROS,ESTUDIO DE FACTIBILIDAD PARA UNA PLANTA RECICL...,I-2004,ysra12@hotmail.com,PROYECTO DE GRADO,IND-002-2004,100,FACTIBILIDAD,Recicladora,"Se cumplieron los objetivos deseados, en los o...",https://drive.google.com/file/d/1JBMpCG3OQSvTS...,No tiene,No tiene,No tiene,FEASIBILITY STUDY FOR A PAPER RECYCLING PLANT ...,The desired objectives were met in the eight c...
3,4,CARLA VIVIAN ROJAS NAVIA,NO CORRESPONDE,NO HAY REGISTROS,DIAGNOSTICO DEL SISTEMA DE COBRANZA INTERNO DE...,I-2004,carlavivianarn@yahoo.es,TRABAJO DIRIGIDO,IND-003-2004,100,SISTEMAS DE INFORMACIÓN,VIVA,La empresa de telecomunicaciones VIVA-GSM cuen...,https://drive.google.com/file/d/1Q1a9v3pwCCkX8...,No tiene,No tiene,No tiene,DIAGNOSIS OF THE INTERNAL COLLECTION SYSTEM OF...,The telecommunications company VIVA-GSM has a ...
4,5,VASQUEZ FLORES YOSHIDA IVETT,NO CORRESPONDE,NO HAY REGISTROS,ESTUDIO DE FACTIBILIDAD PARA LA IMPLEMENTACIÓN...,I-2004,yoshi_112000@yahoo.es,PROYECTO DE GRADO,IND-004-2004,100,FACTIBILIDAD,Yuca,El objetivo principal de este proyecto fue el ...,https://drive.google.com/file/d/1T5xhIdT3JMlGN...,No tiene,No tiene,No tiene,FEASIBILITY STUDY FOR THE IMPLEMENTATION OF A ...,The main objective of this project was to carr...


In [0]:
def limpiar_texto(t):
    t = t.lower()
    t = re.sub(r"[^\w\s]", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

In [0]:
# select thesis titles

list_titles = df_all_trans['thesis_title'].values.tolist()
list_titles_es = df_all_trans['Titulo'].values.tolist()

In [0]:
# clear data

stop_words = set(stopwords.words("english"))

list_titles_stop = [" ".join([word.lower() for word in titulo.split() if word.lower() not in stop_words])
                          for titulo in list_titles]
                          
list_titles_stop = [limpiar_texto(x) for x in list_titles_stop]

In [0]:
# view 5 thesis names

list_titles_stop[0:5]

['feasibility project implementation biodiesel processing plant industrias admsao sa',
 'economic financial feasibility project installation camu camu processing plant',
 'feasibility study paper recycling plant within umss',
 'diagnosis internal collection system financial services unit vivagsm company',
 'feasibility study implementation plant produce sweet cassava starch export']

## max_features = None, ngram_range=(1, 2)

In [0]:
# create model by default, (2) n-grams

vectorizer = TfidfVectorizer(ngram_range=(1, 2)) #  max_features=50
X = vectorizer.fit_transform(list_titles_stop)

In [0]:
# create tf-idf matrix

df_tf_idf_titles = pd.DataFrame(columns =  vectorizer.get_feature_names_out(), data = X.toarray(), index = list_titles_stop)

In [0]:
# matrix dimension

df_tf_idf_titles.shape

(1172, 8806)

In [0]:
# RAM usage

df_tf_idf_titles.info() # 78.7+ MB

<class 'pandas.core.frame.DataFrame'>
Index: 1172 entries, feasibility project implementation biodiesel processing plant industrias admsao sa to feasibility study production marketing fruit coffee chañar metropolitan axis cochabamba
Columns: 8806 entries, 000 to álvarez company
dtypes: float64(8806)
memory usage: 78.7+ MB


In [0]:
df_tf_idf_titles.head(2)

,000,000 yuraj,00918,00918 company,00918 manufacturing,00918 nts,00918 standard,00918 teknolift,038,038 company,03801,10w30,10w30 api,1211002,1211002 ceramic,130062005,130062005 standard,14,14 de,14000,14001,14001 ohsas,14001 standard,14001 standards,140012004,140012004 ohsas,140012004 pilplast,140012004 standard,140012015,140012015 standard,16998,16998ra,16998ra 038,17025,17025 standard,170252005,170252005 capnumss,18001,18001 2008,18001 company,...,yeast taquiña,yesobol,yesobol srl,yogort,yogort processing,yogures,yogures line,yogurt,yogurt cheese,yogurt enriched,yogurt mix,yogurt processing,yogurt production,youth,youth clothing,ypf,ypfb,ypfb flamagas,ypfb industrial,ypfb logísticaplanta,ypfb lubricants,ypfb refinación,ypfb refining,ypfb vallehermoso,yuqui,yuqui ciri,yuquiciri,yuquiciri tco,yuraj,yuraj sub,zambrana,zambrana microenterprise,zedron,zedron orange,zelada,zelada bakery,zinc,zinc fbm,álvarez,álvarez company
feasibility project implementation biodiesel processing plant industrias admsao sa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
economic financial feasibility project installation camu camu processing plant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Similitud por cosenos

![image_1777070277026.png](./image_1777070277026.png "image_1777070277026.png")

In [0]:
def search_similar_documents(query,vectorizer,X,list_titles_en, list_titles_es, k = 5):

    query_vec = vectorizer.transform([query])

    similarities = cosine_similarity(query_vec, X)[0]

    top_indices = np.argsort(similarities)[::-1][:k]

    for i in top_indices:
        print(f"Score: {similarities[i]:.4f}")
        print(f"Documento (EN): {list_titles_en[i]}")
        print(f"Documento (ES): {list_titles_es[i]}")
        print("-" * 50, end = '\n\n')

    return top_indices

In [0]:
# search test 1 

query = 'academic plan'

search_similar_documents(query,vectorizer,X,list_titles, list_titles_es)

Score: 0.3097
Documento (EN): ACADEMIC DEVELOPMENT PLAN FOR THE ELECTROMECHANICAL ENGINEERING CAREER 2019 – 2023
Documento (ES): PLAN DE DESARROLLO ACADÉMICO DE LA CARRERA DE INGENIERÍA ELECTROMECÁNICA 2019 – 2023
--------------------------------------------------

Score: 0.0739
Documento (EN): DESIGN OF AN OCCUPATIONAL SAFETY AND HYGIENE PLAN
Documento (ES): DISEÑO DE UN PLAN DE SEGURIDAD E HIGIENE OCUPACIONAL
--------------------------------------------------

Score: 0.0572
Documento (EN): DESIGN OF A FOOD SAFETY PLAN BASED ON HACCP AND AN INDUSTRIAL SAFETY AND OCCUPATIONAL HYGIENE PLAN FOR THE COMPANY INDUSTRIAS DEL MOLLE S.R.L.
Documento (ES): DISEÑO DE UN PLAN DE INOCUIDAD ALIMENTARIA BASADA EN EL HACCP Y UN PLAN DE SEGURIDAD INDUSTRIAL E HIGIENE OCUPACIONAL PARA LA EMPRESA INDUSTRIAS DEL MOLLE S.R.L
--------------------------------------------------

Score: 0.0547
Documento (EN): STRATEGIC MARKETING PLAN FOR THE VANGUARD COMPANY
Documento (ES): PLAN ESTRATÉGICO DE MARKETING PARA 

array([1020,  984,  461,  352,  525])

In [0]:
# search test 2 

query = 'ypfb'

search_similar_documents(query,vectorizer,X,list_titles, list_titles_es)

Score: 0.3029
Documento (EN): INDUSTRIAL SAFETY AND HYGIENE PLAN FOR THE COMPANY “YPFB FLAMAGAS S.A.”
Documento (ES): PLAN DE SEGURIDAD E HIGIENE INDUSTRIAL PARA LA EMPRESA  “YPFB FLAMAGAS S.A.
--------------------------------------------------

Score: 0.2797
Documento (EN): STRATEGIC MARKETING PLAN FOR THE NEW LINE OF YPFB INDUSTRIAL LUBRICANTS.
Documento (ES): PLAN ESTRATÉGICO DE MARKETING PARA LA NUEVA LÍNEA DE LUBRICANTES INDUSTRIALES YPFB.
--------------------------------------------------

Score: 0.2385
Documento (EN): EVALUATION AND DEVELOPMENT OF IMPROVEMENTS OF THE OPERATIONAL PROCESSES IN “YPFB LOGÍSTICA-PLANTA COCHABAMBA”
Documento (ES): EVALUACIÓN Y DESARROLLO DE MEJORAS DE LOS PROCESOS OPERACIONALES EN ”YPFB LOGÍSTICA-PLANTA COCHABAMBA”
--------------------------------------------------

Score: 0.2108
Documento (EN): PREFEASIBILITY STUDY FOR A DAP/NPK MINERAL FERTILIZER PRODUCTION PLANT IN BOLIVIA, FOR THE INDUSTRIALIZATION MANAGEMENT OF YPFB
Documento (ES): ESTUDIO DE PRE

array([ 820, 1084,  373,  267,  450])

## max_features = 50

In [0]:
# create model by default, (2) n-grams

vectorizer_50 = TfidfVectorizer(ngram_range=(1, 2),max_features=50) #  max_features=50
X_50 = vectorizer_50.fit_transform(list_titles_stop)

In [0]:
# create tf-idf matrix

df_tf_idf_titles_50 = pd.DataFrame(columns =  vectorizer_50.get_feature_names_out(), data = X_50.toarray(), index = list_titles_stop)

In [0]:
# vectorizer_50.get_feature_names_out()[0:500]

In [0]:
df_tf_idf_titles_50.head(2)

,based,based iso,city,cochabamba,company,control,department,department cochabamba,design,development,feasibility,feasibility project,feasibility study,health,hygiene,hygiene plan,implementation,industrial,industrial safety,installation,iso,line,ltda,management,management system,marketing,marketing plan,model,occupational,plan,planning,plant,process,processing,processing plant,production,project,project implementation,quality,quality management,sa,safety,srl,standard,strategic,study,study implementation,system,system based,technical
feasibility project implementation biodiesel processing plant industrias admsao sa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.231787,0.358178,0.0,0.0,0.0,0.0,0.266161,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.263755,0.0,0.36931,0.381741,0.0,0.345507,0.41203,0.0,0.0,0.326059,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
economic financial feasibility project installation camu camu processing plant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.255523,0.394858,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.454224,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.290765,0.0,0.40713,0.420833,0.0,0.380889,0.00000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [0]:
df_tf_idf_titles_50.shape

(1172, 50)

In [0]:
# RAM usage

df_tf_idf_titles_50.info() # 467.0+ KB

<class 'pandas.core.frame.DataFrame'>
Index: 1172 entries, feasibility project implementation biodiesel processing plant industrias admsao sa to feasibility study production marketing fruit coffee chañar metropolitan axis cochabamba
Data columns (total 50 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   based                   1172 non-null   float64
 1   based iso               1172 non-null   float64
 2   city                    1172 non-null   float64
 3   cochabamba              1172 non-null   float64
 4   company                 1172 non-null   float64
 5   control                 1172 non-null   float64
 6   department              1172 non-null   float64
 7   department cochabamba   1172 non-null   float64
 8   design                  1172 non-null   float64
 9   development             1172 non-null   float64
 10  feasibility             1172 non-null   float64
 11  feasibility project     1172 non-null   

In [0]:
# search test 3 

query = "academic plan"

search_similar_documents(query, vectorizer_50,X_50,list_titles_stop, list_titles_es)

Score: 1.0000
Documento (EN): business plan intensive breeding castilla rabbits province punata
Documento (ES): PLAN DE NEGOCIOS PARA LA CRIANZA INTENSIVA DE CONEJOS DE CASTILLA EN LA PROVINCIA DE PUNATA
--------------------------------------------------

Score: 1.0000
Documento (EN): business plan export wooden interior furniture united states market
Documento (ES): PLAN DE NEGOCIOS PARA LA EXPORTACIÓN DE MUEBLES INTERIORES DE MADERA AL MERCADO DE LOS ESTADOS UNIDOS
--------------------------------------------------

Score: 1.0000
Documento (EN): modernization plan quiroz bakery
Documento (ES): PLAN DE MODERNIZACIÓN DE LA PANADERÍA QUIROZ
--------------------------------------------------

Score: 1.0000
Documento (EN): plan conformation textiles clothing cluster faculty sciences technology umss
Documento (ES): PLAN PARA LA CONFORMACIÓN DE UN CLÚSTER DE TEXTILES Y CONFECCIONES EN LA FACULTAD DE CIENCIAS Y TECNOLOGÍA DE LA UMSS
--------------------------------------------------

Score: 

array([699, 190,  74, 994, 356])

In [0]:
# search test 4

query = "ypfb"

search_similar_documents(query, vectorizer_50,X_50,list_titles_stop, list_titles_es)

Score: 0.0000
Documento (EN): feasibility study production marketing fruit coffee chañar metropolitan axis cochabamba
Documento (ES): ESTUDIO DE FACTIBILIDAD PARA LA PRODUCCIÓN Y COMERCIALIZACIÓN DE CAFÉ DEL FRUTO
                  DE CHAÑAR EN EL EJE METROPOLITANO DE COCHABAMBA
--------------------------------------------------

Score: 0.0000
Documento (EN): industrial safety management plan inquibol company
Documento (ES): PLAN DE GESTIÓN DE SEGURIDAD INDUSTRIAL PARA LA EMPRESA INQUIBOL.
--------------------------------------------------

Score: 0.0000
Documento (EN): identification evaluation occupational risks municipal drinking water sewer service company semapa
Documento (ES): IDENTIFICACIÓN Y EVALUACIÓN DE RIESGOS LABORALES PARA LA EMPRESA DE SERVICIO MUNICIPAL DE AGUA POTABLE Y ALCANTARILLADO SEMAPA.
--------------------------------------------------

Score: 0.0000
Documento (EN): development implementation methodology improve efficiency embol sa plant cochabamba
Documento (ES)

array([1171,  401,  385,  386,  387])

# Search engine - abstract

In [0]:
df_all_trans.head(1)

,N,Autor 1,Autor 2,Tutor,Titulo,Semestre,email,Modalidad,Codigo,Calificacion,Categoria,Palabras clave,Resumen,Enlace tesis,Enlace Resumen,Diapositivas,Simulacion,thesis_title,summarize_thesis
0,1,DANNY MARIO OROPEZA ROJAS,NO CORRESPONDE,NO HAY REGISTROS,PROYECTO DE FACTIBILIDAD PARA LA IMPLEMENTACIÓ...,I-2004,dannymario_oropeza@hotmail.com,PROYECTO DE GRADO,IND-000-2004,100,FACTIBILIDAD,Biodiesel,El Proyecto de grado tiene como principal obje...,https://drive.google.com/file/d/1yNmUbZT_nqRNB...,https://drive.google.com/file/d/1TlxFBwnFfYsZW...,NaN,No tiene,FEASIBILITY PROJECT FOR THE IMPLEMENTATION OF ...,The main objective of the degree project is to...


In [0]:
df_all_trans.isna().sum()

N                   0
Autor 1             0
Autor 2             0
Tutor               0
Titulo              0
Semestre            0
email               0
Modalidad           1
Codigo              0
Calificacion        0
Categoria           0
Palabras clave      0
Resumen             1
Enlace tesis        0
Enlace Resumen      4
Diapositivas        1
Simulacion          0
thesis_title        0
summarize_thesis    1
dtype: int64

In [0]:
df_all_trans_clear = df_all_trans.dropna()

In [0]:
df_all_trans_clear.isna().sum()

N                   0
Autor 1             0
Autor 2             0
Tutor               0
Titulo              0
Semestre            0
email               0
Modalidad           0
Codigo              0
Calificacion        0
Categoria           0
Palabras clave      0
Resumen             0
Enlace tesis        0
Enlace Resumen      0
Diapositivas        0
Simulacion          0
thesis_title        0
summarize_thesis    0
dtype: int64

In [0]:
# select thesis abstracts

list_abstract = df_all_trans_clear['summarize_thesis'].values.tolist()
list_abstract_es = df_all_trans_clear['Resumen'].values.tolist()

In [0]:
# select thesis title

list_title_clear= df_all_trans_clear['thesis_title'].values.tolist()
list_title_clear_es = df_all_trans_clear['Titulo'].values.tolist()

In [0]:
list_abstract_stop = [" ".join([word.lower() for word in titulo.split() if word.lower() not in stop_words])
                          for titulo in list_abstract]
                          
list_abstract_stop = [limpiar_texto(x) for x in list_abstract_stop]

## max_features = None

In [0]:
# create model by default, (2) n-grams

vectorizer_abs = TfidfVectorizer(ngram_range=(1, 2)) #  max_features=50
X_abs = vectorizer_abs.fit_transform(list_abstract_stop)

In [0]:
# create tf-idf matrix

df_tf_idf_abs = pd.DataFrame(columns = vectorizer_abs.get_feature_names_out(), data = X_abs.toarray(), index = list_abstract_stop)

In [0]:
df_tf_idf_abs.shape

(1165, 151960)

In [0]:

# RAM usage 

df_tf_idf_abs.info() # memory usage: 1.3+ GB

<class 'pandas.core.frame.DataFrame'>
Index: 1165 entries, camu camu variety fruit origin banks rivers peruvian amazon that due peculiar characteristics recent introduction world markets promises promising crop research carried institute nutrition peru national agrarian university la molina unalm demonstrated high content ascorbic acid camucamu pulp importance economic value mainly due high content ascorbic acid vitamin c element used pharmacological industry natural source vitamin c comparatively camucamu 30 60 times ascorbic acid oranges lemons therefore cultivation camu camu cochabamba tropics constitutes efficient alternative planting illicit crops given within alternative development programs market study divided into market research jam fruit tea based surveys directed homes points sale camucamu pulp market research aimed local foreign market tasting test camucamu fruit jam pulp observed factors influenced determination size mainly characteristics consumer market is demand produc

Dentro de la arquitectura de despliegue, hay que cuidad la latencia(velocidad) y volumen

![image_1777069862282.png](./image_1777069862282.png "image_1777069862282.png")

In [0]:
# search test 5

query = "academic plan"

search_similar_documents(query, vectorizer_abs,X_abs,list_title_clear, list_title_clear_es)

Score: 0.1753
Documento (EN): ACADEMIC DEVELOPMENT PLAN FOR THE ELECTROMECHANICAL ENGINEERING CAREER 2019 – 2023
Documento (ES): PLAN DE DESARROLLO ACADÉMICO DE LA CARRERA DE INGENIERÍA ELECTROMECÁNICA 2019 – 2023
--------------------------------------------------

Score: 0.1378
Documento (EN): CURRICULAR TRANSFORMATION PROJECT FOR THE DEGREE COURSE IN INDUSTRIAL ENGINEERING OF THE FACULTY OF SCIENCES AND TECHNOLOGY OF THE UNIVERSIDAD MAYOR DE SAN SIMÓN
Documento (ES): PROYECTO DE TRANSFORMACIÓN CURRICULAR PARA LA CARRERA LICENCIATURA EN INGENIERÍA INDUSTRIAL DE LA FACULTAD DE  CIENCIAS Y TECNOLOGÍA DE LA UNIVERSIDAD MAYOR DE SAN SIMÓN
--------------------------------------------------

Score: 0.1271
Documento (EN): STRATEGIC DEVELOPMENT PLAN FOR THE SYSTEMS ENGINEERING COURSE OF THE FACULTY OF SCIENCES AND TECHNOLOGY OF THE UNIVERSIDAD MAYOR DE SAN SIMÓN 2019-2023
Documento (ES): PLAN ESTRATÉGICO DE DESARROLLO PARA LA CARRERA DE INGENIERÍA DE SISTEMAS DE LA FACULTAD DE CIENCIAS Y TECN

array([1013,  967, 1000,  947,  957])

Asi como se puede usar el algoritmo para la busqueda, igual puede ser de ayuda para "Sistemas de recomendacion" por similutd de items

![image_1777070050894.png](./image_1777070050894.png "image_1777070050894.png")

In [0]:
# search test 6

query = "ypfb"

search_similar_documents(query, vectorizer_abs,X_abs,list_title_clear, list_title_clear_es)

Score: 0.1866
Documento (EN): DEVELOPMENT OF A STRATEGIC MARKETING PLAN FOR THE NEW SAE 10W30 API SN OIL LINE OF THE YPFB LUBRICANTS BRAND
Documento (ES): DESARROLLO DE UN PLAN ESTRATÉGICO DE MARKETING PARA LA NUEVA LÍNEA DE ACEITE SAE 10W30 API SN DE LA MARCA YPFB LUBRICANTES
--------------------------------------------------

Score: 0.1700
Documento (EN): INDUSTRIAL SAFETY AND HYGIENE PLAN FOR THE COMPANY “YPFB FLAMAGAS S.A.”
Documento (ES): PLAN DE SEGURIDAD E HIGIENE INDUSTRIAL PARA LA EMPRESA  “YPFB FLAMAGAS S.A.
--------------------------------------------------

Score: 0.0785
Documento (EN): FEASIBILITY STUDY TO IMPLEMENT A HOUSEHOLD GAS NETWORK INSTALLATION COMPANY IN COCHABAMBA
Documento (ES): ESTUDIO DE FACTIBILIDAD PARA IMPLANTAR UNA EMPRESA DE INSTALACIÓN DE REDES DE GAS DOMICILIARIO EN COCHABAMBA
--------------------------------------------------

Score: 0.0784
Documento (EN): EVALUATION AND DEVELOPMENT OF IMPROVEMENTS OF THE OPERATIONAL PROCESSES IN “YPFB LOGÍSTICA-PLANTA

array([723, 814, 730, 370, 265])

In [0]:
# search test 7

query = "digital library"

search_similar_documents(query, vectorizer_abs,X_abs,list_title_clear, list_title_clear_es)

Score: 0.2219
Documento (EN): DEVELOPMENT OF A DIGITAL LIBRARY IN THE INDUSTRIAL ENGINEERING COURSE
Documento (ES): DESARROLLO DE UNA BIBLIOTECA DIGITAL EN LA CARRERA DE INGENIERÍA INDUSTRIAL
--------------------------------------------------

Score: 0.0909
Documento (EN): STRATEGIC MARKETING PLAN FOR THE DIGITAL MARKETING COMPANY “SOCIAL BRAND” FOR SMEs IN THE CITY OF COCHABAMBA.”
Documento (ES): PLAN ESTRATÉGICO DE MARKETING PARA LA EMPRESA DE MARKETING DIGITAL “SOCIAL BRAND” PARA PYMES EN LA CIUDAD DE COCHABAMBA”.
--------------------------------------------------

Score: 0.0633
Documento (EN): BUSINESS PLAN FOR THE EXPANSION OF REPAIR AND MAINTENANCE SERVICES OF ELECTRONIC AND DIGITAL HOME EQUIPMENT FOR THE COMPANY SISTEC (ELECTRICAL TECHNICAL SYSTEMS).
Documento (ES): PLAN DE NEGOCIOS PARA LA AMPLIACION DE SERVICIOS DE REPARACION Y           MANTENIMIENTO DE EQUIPOS ELECTRONICOS Y DIGITALES DEL HOGAR PARA LA EMPRESA SISTEC (SISTEMAS TÉCNICOS ELÉCTRICOS).
--------------------------

array([ 674, 1118,  484,  488,  521])

Por nuestra experiencia podemos determinar si los output tienen logica

![image_1777069719076.png](./image_1777069719076.png "image_1777069719076.png")